# 🛍️ Retail Sales Data Analysis, KPI Dashboard & AI-Powered Insights

**A beginner-friendly, end-to-end data analysis notebook for Google Colab**

This notebook was built as a Junior Data Analyst portfolio project. It automatically adapts to any retail
sales dataset (CSV or Excel) you upload — it does **not** assume fixed column names.

**What this notebook does:**
1. Upload & inspect your dataset
2. Clean and validate the data
3. Run exploratory data analysis (EDA)
4. Calculate 5 core KPIs
5. Analyze category, channel, discount, outlier, and customer performance
6. Generate evidence-based business insights
7. Let you ask questions about your data interactively
8. Produce a final business report

> ℹ️ Run the cells **from top to bottom**. Each section explains what it does and why.


---
## 1. Setup & Dataset Upload

Run the cell below once to import all the libraries this notebook needs.


In [ ]:
# Install/import required libraries (all are pre-installed in Google Colab)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print("✅ Libraries loaded successfully.")


### 1.1 Upload your dataset

Click the "Choose Files" button below to upload your retail sales dataset from your computer.
Both **CSV** and **Excel** (.xlsx/.xls) files are supported. If you upload more than one file,
you'll be asked to pick which one to analyze.

> This cell uses Google Colab's native `files.upload()` — no external web frameworks are used.


In [ ]:
# --- Google Colab native file upload ---
# If this notebook is NOT running in Colab (e.g. local Jupyter), it will fall back
# to looking for a CSV/Excel file already sitting in the working directory.

uploaded_files = {}

try:
    from google.colab import files
    print("Please choose a CSV or Excel file to upload...")
    uploaded = files.upload()
    uploaded_files = uploaded  # dict of filename -> bytes
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab — looking for a local CSV/Excel file instead.")
    import glob
    local_candidates = glob.glob('*.csv') + glob.glob('*.xlsx') + glob.glob('*.xls')
    for f in local_candidates:
        with open(f, 'rb') as fh:
            uploaded_files[f] = fh.read()
    if not uploaded_files:
        print("⚠️ No dataset found. Please place a .csv or .xlsx file in the working "
              "directory and re-run this cell.")

print()
print(f"Files detected: {list(uploaded_files.keys())}")


In [ ]:
# --- Identify supported dataset files and let the user pick one if needed ---
import io

supported = [f for f in uploaded_files if f.lower().endswith(('.csv', '.xlsx', '.xls'))]

if not supported:
    raise ValueError("No supported CSV/Excel file was uploaded. Please re-run the upload cell.")

if len(supported) == 1:
    selected_file = supported[0]
    print(f"Using the only supported file uploaded: '{selected_file}'")
else:
    print("Multiple supported files were uploaded:")
    for i, f in enumerate(supported):
        print(f"  [{i}] {f}")
    idx = 0  # Default to the first file. Change this index if you want a different one.
    print(f"\n➡️ Defaulting to file index {idx}: '{supported[idx]}'. "
          f"Edit the 'idx' variable above and re-run this cell to pick a different file.")
    selected_file = supported[idx]

# --- Load the dataset into a DataFrame ---
raw_bytes = uploaded_files[selected_file]

if selected_file.lower().endswith('.csv'):
    df_raw = pd.read_csv(io.BytesIO(raw_bytes))
else:
    df_raw = pd.read_excel(io.BytesIO(raw_bytes))

print(f"\n✅ Loaded '{selected_file}' successfully.")


In [ ]:
# --- Basic file & dataset info ---
print(f"📄 Filename: {selected_file}")
print(f"📐 Dimensions: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
print("Column names and data types:")
dtype_table = pd.DataFrame({
    'column': df_raw.columns,
    'dtype': df_raw.dtypes.astype(str).values,
    'non_null_count': df_raw.notna().sum().values,
    'unique_values': [df_raw[c].nunique() for c in df_raw.columns]
})
dtype_table


---
## 2. Data Understanding

Before cleaning or analyzing anything, we automatically classify each column by type and try to
identify which columns represent transaction IDs, dates, categories, quantities, prices, discounts,
channels, customers, etc. This detection is **keyword-based**, so the notebook works on datasets
with different column names — you are not locked into one fixed schema.


In [ ]:
# --- Classify columns by data type ---
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_raw.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
datetime_cols = df_raw.select_dtypes(include=['datetime64']).columns.tolist()

# Try to auto-detect date-like columns stored as text
candidate_date_cols = []
for c in categorical_cols:
    if 'date' in c.lower():
        candidate_date_cols.append(c)

print(f"🔢 Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"🔤 Categorical/text columns ({len(categorical_cols)}): {categorical_cols}")
print(f"📅 Already-datetime columns ({len(datetime_cols)}): {datetime_cols}")
print(f"📅 Candidate date columns stored as text: {candidate_date_cols}")


In [ ]:
# --- Auto-detect the business meaning of each column via keyword matching ---
def detect_columns(df):
    """Guess which column plays which business role, based on column-name keywords.
    Returns a dict mapping role -> actual column name (or None if not found)."""
    cols_lower = {c.lower(): c for c in df.columns}

    def find(*keywords):
        for kw in keywords:
            for lc, orig in cols_lower.items():
                if kw in lc:
                    return orig
        return None

    return {
        'transaction_id': find('transaction_id', 'order_id', 'invoice_id', 'invoice_no', 'txn_id'),
        'date':           find('date', 'order_date', 'transaction_date', 'purchase_date'),
        'category':       find('product_category', 'category', 'product_type'),
        'quantity':       find('units_sold', 'quantity', 'qty', 'units'),
        'unit_price':     find('unit_price', 'price_per_unit', 'unitprice', 'item_price'),
        'revenue':        find('revenue', 'total_amount', 'total_sales', 'sales_amount', 'amount', 'sales'),
        'discount':       find('discount_pct', 'discount_rate', 'discount'),
        'channel':        find('sales_channel', 'channel', 'platform'),
        'customer_id':    find('customer_id', 'cust_id', 'client_id'),
        'region':         find('region', 'store_location', 'location'),
        'store':          find('store_id', 'store_name', 'store', 'branch'),
        'segment':        find('customer_segment', 'segment', 'customer_type'),
        'product_id':     find('sku', 'product_id', 'item_id'),
    }

detected = detect_columns(df_raw)

print("🔎 Detected column roles:\n")
for role, col in detected.items():
    status = f"'{col}'" if col else "❌ not found"
    print(f"  {role:15s} -> {status}")


In [ ]:
# --- Determine whether this is one-row-per-transaction or multi-row-per-order data ---
if detected['transaction_id']:
    n_rows = len(df_raw)
    n_unique_ids = df_raw[detected['transaction_id']].nunique()
    if n_unique_ids == n_rows:
        grain = "ONE ROW PER TRANSACTION (transaction_id is unique per row)"
    else:
        avg_rows_per_id = n_rows / n_unique_ids
        grain = (f"MULTIPLE ROWS PER ORDER (avg {avg_rows_per_id:.2f} product lines per order — "
                 f"likely one row per product within an order)")
    print(f"Rows: {n_rows:,} | Unique transaction IDs: {n_unique_ids:,}")
    print(f"➡️ Dataset grain: {grain}")
else:
    print("⚠️ No transaction/order ID column detected — cannot determine dataset grain. "
          "Each row will be treated as one transaction for KPI purposes.")


In [ ]:
# --- Flag potential naming/formatting inconsistencies for manual awareness ---
print("Potential formatting inconsistencies to review:\n")

for c in categorical_cols:
    sample_vals = df_raw[c].dropna().astype(str).unique()[:10]
    has_mixed_case = len(set(v.lower() for v in sample_vals)) < len(set(sample_vals))
    has_whitespace = any(v != v.strip() for v in sample_vals)
    if has_mixed_case or has_whitespace:
        print(f"  '{c}': possible inconsistent casing/whitespace in values like {list(sample_vals)[:5]}")

print("\n(If nothing printed above, no obvious naming inconsistencies were detected.)")


---
## 3. Data Cleaning & Preprocessing

We keep the original uploaded data untouched in `df_raw` and perform all cleaning on a copy, `df`,
so you can always compare before/after.


In [ ]:
# Preserve the original dataset for comparison
df = df_raw.copy()
original_row_count = len(df)
cleaning_log = []  # we'll record every cleaning action taken, for the final summary report


### 3.1 Missing Values

In [ ]:
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing_counts, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_pct', ascending=False)

if missing_report.empty:
    print("✅ No missing values found in any column.")
    cleaning_log.append("No missing values were present in the dataset.")
else:
    print("Columns with missing values:")
    display(missing_report)

    # Business-meaning-aware handling (not blind zero-fill):
    for col in missing_report.index:
        pct = missing_report.loc[col, 'missing_pct']
        if col in numeric_cols:
            if col == detected.get('discount'):
                # Missing discount most plausibly means "no discount applied"
                df[col] = df[col].fillna(0)
                cleaning_log.append(f"'{col}': missing values filled with 0 (assumed: no discount applied).")
            elif col in (detected.get('quantity'), detected.get('revenue'), detected.get('unit_price')):
                # Missing quantity/revenue/price is a data-quality issue, not "zero" -
                # dropping is safer than fabricating a transaction value.
                before = len(df)
                df = df.dropna(subset=[col])
                cleaning_log.append(f"'{col}': {before - len(df)} rows dropped "
                                     f"(missing core transaction value; imputing would fabricate revenue).")
            else:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                cleaning_log.append(f"'{col}': missing values filled with median ({median_val}).")
        else:
            df[col] = df[col].fillna('Unknown')
            cleaning_log.append(f"'{col}': missing values filled with 'Unknown' category.")

    print("\n✅ Missing values handled per-column, based on business meaning (see cleaning_log).")


### 3.2 Duplicate Records

In [ ]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

if exact_dupes > 0:
    before = len(df)
    df = df.drop_duplicates()
    cleaning_log.append(f"Removed {before - len(df)} exact duplicate rows.")
    print(f"✅ Removed {before - len(df)} exact duplicate rows.")
else:
    cleaning_log.append("No exact duplicate rows found.")

if detected['transaction_id']:
    dup_ids = df[detected['transaction_id']].duplicated().sum()
    print(f"Rows sharing a duplicate transaction/order ID: {dup_ids}")
    if dup_ids > 0:
        print("ℹ️ These are NOT automatically removed — a shared order ID with different "
              "product/SKU rows may legitimately represent multiple items in one order.")
        cleaning_log.append(f"{dup_ids} rows share a transaction ID with another row; "
                             f"kept as-is (likely multi-item orders, not exact duplicates).")


### 3.3 Data Type Correction

In [ ]:
# --- Convert the date column to datetime ---
dtc = detected['date']
if dtc:
    before_na = df[dtc].isna().sum()
    df[dtc] = pd.to_datetime(df[dtc], errors='coerce')
    after_na = df[dtc].isna().sum()
    invalid_dates = after_na - before_na
    if invalid_dates > 0:
        cleaning_log.append(f"'{dtc}': {invalid_dates} values could not be parsed as dates and became NaT.")
    else:
        cleaning_log.append(f"'{dtc}': converted to datetime successfully, no invalid dates found.")
    print(f"✅ '{dtc}' converted to datetime. Range: {df[dtc].min()} to {df[dtc].max()}")
else:
    print("⚠️ No date column detected — time-series analysis will be skipped later.")

# --- Clean & coerce numeric columns (strip currency symbols, commas, spaces) ---
for role in ['quantity', 'revenue', 'unit_price', 'discount']:
    c = detected[role]
    if c and df[c].dtype == object:
        df[c] = (df[c].astype(str)
                       .str.replace(r'[^\d.\-]', '', regex=True)
                       .replace('', np.nan))
        df[c] = pd.to_numeric(df[c], errors='coerce')
        cleaning_log.append(f"'{c}': cleaned currency symbols/commas and converted to numeric.")

# --- Standardize categorical text values (trim whitespace, consistent casing for comparison) ---
for c in categorical_cols:
    if c in df.columns and df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip()

print("✅ Data type correction complete.")


### 3.4 Invalid Values

In [ ]:
qc, rc, pc, dc = detected['quantity'], detected['revenue'], detected['unit_price'], detected['discount']
invalid_flags = {}

if qc:
    invalid_flags['negative_quantity'] = int((df[qc] < 0).sum())
if rc:
    invalid_flags['zero_or_negative_revenue'] = int((df[rc] <= 0).sum())
if pc:
    invalid_flags['zero_or_negative_unit_price'] = int((df[pc] <= 0).sum())
if dc:
    invalid_flags['negative_discount'] = int((df[dc] < 0).sum())
    invalid_flags['discount_over_100pct'] = int((df[dc] > 100).sum())
if dtc:
    invalid_flags['invalid_dates'] = int(df[dtc].isna().sum())

print("Invalid-value checks (flagged, NOT auto-deleted):")
for k, v in invalid_flags.items():
    flag = "⚠️" if v > 0 else "✅"
    print(f"  {flag} {k}: {v}")

print("\nℹ️ These records are flagged for review rather than automatically removed, since "
      "e.g. a negative quantity could legitimately represent a return/refund.")
cleaning_log.append(f"Invalid-value scan: {invalid_flags}")


### 3.5 Cleaning Summary Report

In [ ]:
final_row_count = len(df)

print("=" * 60)
print("CLEANING SUMMARY REPORT")
print("=" * 60)
print(f"Original row count : {original_row_count:,}")
print(f"Final row count    : {final_row_count:,}")
print(f"Rows removed       : {original_row_count - final_row_count:,}")
print("-" * 60)
print("Actions taken:")
for i, entry in enumerate(cleaning_log, 1):
    print(f"  {i}. {entry}")
print("=" * 60)
print("\nThe original upload remains available (untouched) in 'df_raw' for comparison.")


---
## 4. Exploratory Data Analysis (EDA)


### 4.1 Statistical Summary

In [ ]:
# Full statistical description of all columns (numeric + categorical)
summary = df.describe(include='all').T
summary


In [ ]:
# Deeper statistics for numeric columns: skewness & kurtosis
numeric_cols_clean = df.select_dtypes(include=[np.number]).columns.tolist()

extra_stats = pd.DataFrame({
    'mean': df[numeric_cols_clean].mean(),
    'median': df[numeric_cols_clean].median(),
    'std': df[numeric_cols_clean].std(),
    'min': df[numeric_cols_clean].min(),
    'max': df[numeric_cols_clean].max(),
    'Q1': df[numeric_cols_clean].quantile(0.25),
    'Q3': df[numeric_cols_clean].quantile(0.75),
    'IQR': df[numeric_cols_clean].quantile(0.75) - df[numeric_cols_clean].quantile(0.25),
    'skewness': df[numeric_cols_clean].skew(),
    'kurtosis': df[numeric_cols_clean].kurt(),
})
extra_stats.round(2)


### 4.2 Univariate Analysis

In [ ]:
# Histograms for numeric columns
n = len(numeric_cols_clean)
fig, axes = plt.subplots(nrows=(n + 2) // 3, ncols=3, figsize=(15, 4 * ((n + 2) // 3)))
axes = axes.flatten() if n > 1 else [axes]

for i, col in enumerate(numeric_cols_clean):
    axes[i].hist(df[col].dropna(), bins=30, color='#4C72B0', edgecolor='white')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Box plots for numeric columns (to visually spot outliers/spread)
fig, axes = plt.subplots(nrows=(n + 2) // 3, ncols=3, figsize=(15, 4 * ((n + 2) // 3)))
axes = axes.flatten() if n > 1 else [axes]

for i, col in enumerate(numeric_cols_clean):
    sns.boxplot(x=df[col], ax=axes[i], color='#55A868')
    axes[i].set_title(f'Box plot: {col}')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Frequency counts for key categorical columns
cat_cols_to_show = [c for c in [detected['category'], detected['channel'], detected['region'],
                                  detected['segment'], detected['store']] if c]

for c in cat_cols_to_show:
    print(f"\n--- {c} ---")
    counts = df[c].value_counts()
    display(counts.to_frame('count'))


### 4.3 Bivariate Analysis

In [ ]:
# Quantity vs Revenue
if qc and rc:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df[qc], df[rc], alpha=0.4, color='#4C72B0')
    ax.set_title(f'{qc} vs {rc}')
    ax.set_xlabel(qc)
    ax.set_ylabel(rc)
    plt.tight_layout()
    plt.show()
    corr = df[[qc, rc]].corr().iloc[0, 1]
    print(f"Correlation ({qc} vs {rc}): {corr:.3f}")
else:
    print("⚠️ Quantity and/or revenue column not found — skipping this analysis.")


In [ ]:
# Unit price vs Revenue (only if a unit price column exists)
if pc and rc:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df[pc], df[rc], alpha=0.4, color='#C44E52')
    ax.set_title(f'{pc} vs {rc}')
    ax.set_xlabel(pc)
    ax.set_ylabel(rc)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No separate unit-price column was detected in this dataset — "
          "'revenue' already represents the final transaction value, so this analysis is skipped.")


In [ ]:
# Discount vs Revenue
if dc and rc:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df[dc], df[rc], alpha=0.4, color='#8172B2')
    ax.set_title(f'{dc} vs {rc}')
    ax.set_xlabel(dc)
    ax.set_ylabel(rc)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Discount and/or revenue column not found — skipping this analysis.")


In [ ]:
# Product category vs Revenue
if detected['category'] and rc:
    cat_rev = df.groupby(detected['category'])[rc].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    cat_rev.plot(kind='bar', ax=ax, color='#4C72B0')
    ax.set_title(f'Total Revenue by {detected["category"]}')
    ax.set_xlabel(detected['category'])
    ax.set_ylabel(rc)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Category and/or revenue column not found — skipping this analysis.")


In [ ]:
# Sales channel vs Revenue
if detected['channel'] and rc:
    ch_rev = df.groupby(detected['channel'])[rc].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 5))
    ch_rev.plot(kind='bar', ax=ax, color='#55A868')
    ax.set_title(f'Total Revenue by {detected["channel"]}')
    ax.set_ylabel(rc)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No sales channel column detected — skipping this analysis.")


In [ ]:
# Correlation heatmap across all numeric columns
if len(numeric_cols_clean) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    corr_matrix = df[numeric_cols_clean].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap (Numeric Columns)')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Not enough numeric columns for a correlation heatmap.")


### 4.4 Time-Series Analysis

In [ ]:
if dtc:
    ts_df = df.dropna(subset=[dtc]).copy()
    ts_df['_day'] = ts_df[dtc].dt.date
    ts_df['_week'] = ts_df[dtc].dt.to_period('W').astype(str)
    ts_df['_month'] = ts_df[dtc].dt.to_period('M').astype(str)

    daily_rev = ts_df.groupby('_day')[rc].sum()
    weekly_rev = ts_df.groupby('_week')[rc].sum()
    monthly_rev = ts_df.groupby('_month')[rc].sum()

    fig, axes = plt.subplots(3, 1, figsize=(13, 12))
    daily_rev.plot(ax=axes[0], color='#4C72B0')
    axes[0].set_title('Daily Revenue')
    axes[0].set_ylabel(rc)

    weekly_rev.plot(ax=axes[1], color='#55A868')
    axes[1].set_title('Weekly Revenue')
    axes[1].set_ylabel(rc)

    monthly_rev.plot(ax=axes[2], kind='line', marker='o', color='#C44E52')
    axes[2].set_title('Monthly Revenue')
    axes[2].set_ylabel(rc)
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    peak_month = monthly_rev.idxmax()
    low_month = monthly_rev.idxmin()
    print(f"📈 Peak revenue month: {peak_month} ({monthly_rev.max():,.2f})")
    print(f"📉 Lowest revenue month: {low_month} ({monthly_rev.min():,.2f})")
else:
    print("⚠️ No valid date column was found — daily/weekly/monthly trend analysis cannot be performed. "
          "All time-based KPIs and insights later in this notebook will be skipped for the same reason.")


In [ ]:
# Month-over-month revenue growth
if dtc:
    mom = monthly_rev.pct_change() * 100
    mom_table = pd.DataFrame({
        'monthly_revenue': monthly_rev,
        'previous_month_revenue': monthly_rev.shift(1),
        'absolute_change': monthly_rev.diff(),
        'pct_growth': mom
    }).round(2)
    display(mom_table)

    positive_months = (mom > 0).sum()
    negative_months = (mom < 0).sum()
    print(f"\n📊 Months with positive growth: {positive_months} | Months with negative growth: {negative_months}")
else:
    print("⚠️ Skipped — no date column available.")


---
## 5. Key Performance Indicators (KPIs)

We calculate 5 core KPIs. Where the dataset already stores a final per-row `revenue` value
(rather than separate quantity × unit price), that is treated as the authoritative net revenue,
so discounts are **not** re-applied on top of it (which would double-count them).


In [ ]:
# KPI 1: Total Revenue
# If a separate unit_price column exists, gross revenue = quantity * unit_price.
# The dataset's own revenue column (if present) is treated as NET revenue (post-discount),
# since re-applying the discount on top of it would double-count the reduction.
if rc:
    net_revenue = df[rc].sum()
    if pc and qc:
        gross_revenue = (df[qc] * df[pc]).sum()
        print(f"Gross Revenue (quantity × unit_price): {gross_revenue:,.2f}")
        print(f"Net Revenue (from '{rc}' column):        {net_revenue:,.2f}")
    else:
        print(f"Total Revenue (from '{rc}' column, treated as final/net transaction value): {net_revenue:,.2f}")
        print("ℹ️ No separate unit-price column exists, so gross vs. net revenue cannot be distinguished; "
              "the revenue column is assumed to already be the final transaction amount.")
    total_revenue = net_revenue
else:
    total_revenue = None
    print("❌ No revenue column detected — Total Revenue cannot be calculated.")


In [ ]:
# KPI 2: Total Orders
if detected['transaction_id']:
    total_orders = df[detected['transaction_id']].nunique()
    print(f"Total Orders (unique transaction IDs): {total_orders:,}")
else:
    total_orders = len(df)
    print(f"⚠️ No transaction/order ID column found. Falling back to row count as a proxy: {total_orders:,} "
          f"(this may overcount orders if the dataset has multiple rows per order).")


In [ ]:
# KPI 3: Average Order Value (AOV) = Total Revenue / Total Orders
if total_revenue is not None and total_orders:
    aov = total_revenue / total_orders
    print(f"Average Order Value (AOV): {aov:,.2f}")
else:
    aov = None
    print("❌ Cannot calculate AOV without both Total Revenue and Total Orders.")


In [ ]:
# KPI 4: Total Units Sold
if qc:
    total_units = df[qc].sum()
    print(f"Total Units Sold: {total_units:,}")
    print("ℹ️ This includes every unit in every row of the cleaned dataset. Negative quantities "
          "(if any were flagged in Section 3.4) would represent returns and are included as-is, "
          "since they were not removed during cleaning.")
else:
    total_units = None
    print("❌ No quantity column detected — Total Units Sold cannot be calculated.")


In [ ]:
# KPI 5: Sales Growth Rate (Month-over-Month)
if dtc and rc:
    growth_table = mom_table.copy()
    # Handle zero previous-period revenue safely (avoid inf/-inf)
    growth_table['pct_growth'] = growth_table['pct_growth'].replace([np.inf, -np.inf], np.nan)
    display(growth_table)

    avg_growth = growth_table['pct_growth'].mean()
    latest_growth = growth_table['pct_growth'].iloc[-1]
    print(f"\nAverage month-over-month growth rate: {avg_growth:.2f}%")
    print(f"Most recent month's growth rate: {latest_growth:.2f}%")
else:
    growth_table = None
    print("❌ Sales Growth Rate requires both a date column and a revenue column — not available.")


In [ ]:
# --- KPI Summary Table ---
kpi_summary = pd.DataFrame([
    {'KPI': 'Total Revenue', 'Formula': 'Σ(Quantity × Unit Price) or Σ(Revenue)',
     'Value': f"{total_revenue:,.2f}" if total_revenue is not None else 'N/A',
     'Interpretation': 'Overall sales value generated in the period covered by the data.'},
    {'KPI': 'Total Orders', 'Formula': 'COUNT DISTINCT(Order/Transaction ID)',
     'Value': f"{total_orders:,}" if total_orders else 'N/A',
     'Interpretation': 'Number of distinct transactions completed.'},
    {'KPI': 'Average Order Value (AOV)', 'Formula': 'Total Revenue / Total Orders',
     'Value': f"{aov:,.2f}" if aov is not None else 'N/A',
     'Interpretation': 'Typical amount a customer spends per order.'},
    {'KPI': 'Total Units Sold', 'Formula': 'Σ(Quantity)',
     'Value': f"{total_units:,}" if total_units is not None else 'N/A',
     'Interpretation': 'Total sales volume in units, independent of price.'},
    {'KPI': 'Avg. Monthly Sales Growth', 'Formula': '((Current − Previous) / Previous) × 100, averaged',
     'Value': f"{avg_growth:.2f}%" if growth_table is not None else 'N/A',
     'Interpretation': 'Average month-over-month revenue trend across the period.'},
])
kpi_summary


In [ ]:
# --- KPI Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kpi_bar_data = {
    'Total Revenue': total_revenue,
    'Total Units Sold': total_units,
}
kpi_bar_data = {k: v for k, v in kpi_bar_data.items() if v is not None}
if kpi_bar_data:
    axes[0].bar(kpi_bar_data.keys(), kpi_bar_data.values(), color=['#4C72B0', '#55A868'])
    axes[0].set_title('Core Volume KPIs')
    for i, v in enumerate(kpi_bar_data.values()):
        axes[0].text(i, v, f'{v:,.0f}', ha='center', va='bottom')

if growth_table is not None:
    axes[1].bar(growth_table.index.astype(str), growth_table['pct_growth'],
                color=['#55A868' if x >= 0 else '#C44E52' for x in growth_table['pct_growth'].fillna(0)])
    axes[1].set_title('Monthly Sales Growth Rate (%)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(0, color='black', linewidth=0.8)
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()


---
## 6. Additional Business Performance Analysis


### 6.A Product Category Performance

In [ ]:
if detected['category'] and rc:
    cat_col = detected['category']
    cat_revenue = df.groupby(cat_col)[rc].sum().sort_values(ascending=False)
    cat_units = df.groupby(cat_col)[qc].sum() if qc else None
    cat_contribution = (cat_revenue / cat_revenue.sum() * 100).round(2)

    cat_table = pd.DataFrame({'total_revenue': cat_revenue, 'revenue_contribution_pct': cat_contribution})
    if cat_units is not None:
        cat_table['units_sold'] = cat_units
        cat_table['avg_selling_price'] = (cat_revenue / cat_units).round(2)

    print("Top 3 categories by revenue:")
    display(cat_table.head(3))
    print("\nBottom 3 categories by revenue:")
    display(cat_table.tail(3))
else:
    print("⚠️ Category and/or revenue column not found — skipping category performance analysis.")


### 6.B Sales Channel Performance

In [ ]:
if detected['channel'] and rc:
    chan_col = detected['channel']
    chan_table = df.groupby(chan_col).agg(
        total_revenue=(rc, 'sum'),
        order_count=(detected['transaction_id'], 'nunique') if detected['transaction_id'] else (rc, 'count'),
        units_sold=(qc, 'sum') if qc else (rc, 'count'),
    )
    chan_table['aov'] = (chan_table['total_revenue'] / chan_table['order_count']).round(2)
    chan_table['revenue_contribution_pct'] = (chan_table['total_revenue'] / chan_table['total_revenue'].sum() * 100).round(2)
    display(chan_table.sort_values('total_revenue', ascending=False))

    fig, ax = plt.subplots(figsize=(7, 5))
    chan_table['total_revenue'].sort_values().plot(kind='barh', ax=ax, color='#4C72B0')
    ax.set_title('Revenue by Sales Channel')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No sales channel column detected — skipping channel performance analysis.")


### 6.C Discount Analysis

In [ ]:
if dc and rc:
    avg_discount = df[dc].mean()
    print(f"Average discount rate: {avg_discount:.2f}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df[dc], df[rc], alpha=0.4, color='#8172B2')
    ax.set_title('Discount vs. Revenue')
    ax.set_xlabel(dc)
    ax.set_ylabel(rc)
    plt.tight_layout()
    plt.show()

    pearson_corr, pearson_p = stats.pearsonr(df[dc], df[rc])
    spearman_corr, spearman_p = stats.spearmanr(df[dc], df[rc])
    print(f"Pearson correlation:  {pearson_corr:.3f} (p={pearson_p:.4f})")
    print(f"Spearman correlation: {spearman_corr:.3f} (p={spearman_p:.4f})")
    print("\n⚠️ Note: correlation does not establish causation. A weak or negative correlation here "
          "does not necessarily mean discounts fail to drive sales — it may reflect which products "
          "or categories tend to be discounted, seasonality, or other confounding factors.")
else:
    print("⚠️ Discount and/or revenue column not found — skipping discount analysis.")


### 6.D Outlier Detection (IQR Method)

In [ ]:
if rc:
    Q1, Q3 = df[rc].quantile(0.25), df[rc].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[rc] < lower_bound) | (df[rc] > upper_bound)]
    print(f"Lower Bound = Q1 - 1.5×IQR = {lower_bound:,.2f}")
    print(f"Upper Bound = Q3 + 1.5×IQR = {upper_bound:,.2f}")
    print(f"Number of outlier transactions on '{rc}': {len(outliers):,} ({len(outliers)/len(df)*100:.1f}% of rows)")

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(x=df[rc], ax=ax, color='#C44E52')
    ax.set_title(f'Outlier Detection: {rc}')
    plt.tight_layout()
    plt.show()

    print("\nSample of flagged outlier transactions (largest 5 by revenue):")
    display(outliers.sort_values(rc, ascending=False).head(5))
    print("\nℹ️ These are NOT removed from the primary analysis — they may represent legitimate "
          "bulk orders, premium products, or wholesale transactions worth investigating rather than errors.")
else:
    print("⚠️ No revenue column detected — skipping outlier detection.")


### 6.E Customer Analysis

In [ ]:
if detected['customer_id'] and rc:
    cust_col = detected['customer_id']
    unique_customers = df[cust_col].nunique()
    cust_revenue = df.groupby(cust_col)[rc].sum().sort_values(ascending=False)

    if detected['transaction_id']:
        cust_order_counts = df.groupby(cust_col)[detected['transaction_id']].nunique()
    else:
        cust_order_counts = df.groupby(cust_col).size()
    repeat_customers = (cust_order_counts > 1).sum()

    avg_spend_per_customer = cust_revenue.mean()

    print(f"Unique customers: {unique_customers:,}")
    print(f"Repeat customers (2+ orders): {repeat_customers:,} ({repeat_customers/unique_customers*100:.1f}%)")
    print(f"Average revenue per customer: {avg_spend_per_customer:,.2f}")
    print("\nTop 10 highest-value customers:")
    display(cust_revenue.head(10).to_frame('total_revenue'))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(cust_revenue, bins=30, color='#4C72B0', edgecolor='white')
    ax.set_title('Distribution of Revenue per Customer')
    ax.set_xlabel('Total revenue per customer')
    ax.set_ylabel('Number of customers')
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ No customer ID column detected — skipping customer-level analysis.")


---
## 7. Business Insights

Every insight below is generated automatically from the values calculated earlier in this notebook —
nothing is hardcoded or fabricated. Each insight states the **Finding**, the **Evidence** (calculated
metric), a **Business Interpretation**, and a **Suggested Action**.


In [ ]:
insights = []

def add_insight(finding, evidence, interpretation, action):
    insights.append({'Finding': finding, 'Evidence': evidence,
                      'Business Interpretation': interpretation, 'Suggested Action': action})

# 1. Overall revenue performance
if total_revenue is not None:
    add_insight(
        f"Total revenue across the dataset is {total_revenue:,.2f}.",
        f"Sum of '{rc}' across {len(df):,} cleaned transaction rows.",
        "This is the overall scale of sales activity captured in the dataset.",
        "Compare this figure against a revenue target or prior-period benchmark, if available."
    )

# 2. Order volume and AOV
if total_orders and aov is not None:
    add_insight(
        f"The business processed {total_orders:,} orders at an average order value of {aov:,.2f}.",
        "Total Orders and AOV KPIs (Section 5).",
        "AOV indicates typical basket size; low AOV relative to revenue could mean high order "
        "volume with small baskets, while high AOV could mean fewer, larger transactions.",
        "Test whether cross-sell or bundling strategies could lift AOV without hurting order volume."
    )

# 3. Monthly revenue growth
if dtc and growth_table is not None:
    best_month = growth_table['pct_growth'].idxmax()
    worst_month = growth_table['pct_growth'].idxmin()
    add_insight(
        f"The strongest month-over-month growth was in {best_month} "
        f"({growth_table['pct_growth'].max():.1f}%); the weakest was in {worst_month} "
        f"({growth_table['pct_growth'].min():.1f}%).",
        "Month-over-month growth table (Section 5, KPI 5).",
        "Volatility between best and worst months suggests revenue may be seasonal or "
        "influenced by promotions/events rather than steady organic growth.",
        "Investigate what drove the best month (promotions, seasonality, new products) "
        "and whether it can be replicated."
    )

# 4 & 5. Top / underperforming categories
if detected['category'] and rc:
    top_cat = cat_table.head(3)
    bottom_cat = cat_table.tail(3)
    add_insight(
        f"The top 3 categories by revenue are {', '.join(top_cat.index.tolist())}, "
        f"contributing {top_cat['revenue_contribution_pct'].sum():.1f}% of total revenue combined.",
        "Category revenue contribution table (Section 6.A).",
        "Revenue is concentrated in a small number of categories, which may indicate strong "
        "product-market fit there, but also revenue concentration risk.",
        "Consider whether inventory, marketing spend, and staffing are appropriately weighted "
        "toward these top categories."
    )
    add_insight(
        f"The bottom 3 categories by revenue are {', '.join(bottom_cat.index.tolist())}, "
        f"contributing only {bottom_cat['revenue_contribution_pct'].sum():.1f}% of total revenue combined.",
        "Category revenue contribution table (Section 6.A).",
        "These categories may be underperforming due to low demand, poor placement, pricing, "
        "or limited assortment — the data alone cannot say which.",
        "Investigate whether to invest further in these categories, reposition them, or "
        "reduce assortment/inventory exposure."
    )

# 6. Sales channel performance
if detected['channel'] and rc:
    leading_channel = chan_table['total_revenue'].idxmax()
    add_insight(
        f"'{leading_channel}' is the leading sales channel by revenue, contributing "
        f"{chan_table.loc[leading_channel, 'revenue_contribution_pct']:.1f}% of total revenue "
        f"with an AOV of {chan_table.loc[leading_channel, 'aov']:,.2f}.",
        "Channel performance table (Section 6.B).",
        "Channel mix shows where customers are currently transacting and at what basket size.",
        "Evaluate whether marketing/operations investment across channels matches their "
        "relative revenue contribution."
    )

# 7. Discount vs revenue relationship
if dc and rc:
    strength = "weak" if abs(pearson_corr) < 0.3 else ("moderate" if abs(pearson_corr) < 0.6 else "strong")
    direction = "positive" if pearson_corr > 0 else "negative"
    add_insight(
        f"The correlation between {dc} and {rc} is {pearson_corr:.2f} (Pearson) — a {strength}, "
        f"{direction} relationship.",
        "Pearson and Spearman correlation coefficients (Section 6.C).",
        "A weak correlation suggests discount depth alone does not strongly predict transaction "
        "revenue in this dataset; other factors (product, category, quantity) likely dominate.",
        "Avoid assuming discounts are the primary revenue driver; test discount effectiveness "
        "with controlled experiments (e.g. A/B tests) rather than relying on this correlation alone."
    )

# 8. Outliers
if rc:
    add_insight(
        f"{len(outliers):,} transactions ({len(outliers)/len(df)*100:.1f}% of all rows) were flagged "
        f"as revenue outliers using the IQR method (bounds: {lower_bound:,.2f} to {upper_bound:,.2f}).",
        "IQR outlier detection (Section 6.D).",
        "These could be legitimate bulk/wholesale orders, premium purchases, or in some cases "
        "data entry errors — the data alone doesn't distinguish between these.",
        "Manually review the largest flagged outliers to confirm they are legitimate, and "
        "consider whether they warrant a separate reporting category (e.g. 'bulk orders')."
    )

# 9. Customer behavior
if detected['customer_id'] and rc:
    add_insight(
        f"{unique_customers:,} unique customers were recorded, of whom {repeat_customers:,} "
        f"({repeat_customers/unique_customers*100:.1f}%) placed more than one order.",
        "Customer analysis (Section 6.E).",
        "A high repeat-customer rate suggests healthy retention; a low rate may indicate the "
        "business is more reliant on acquiring new customers than retaining existing ones.",
        "If the repeat rate is low, consider loyalty programs or retention campaigns; if high, "
        "consider referral programs to leverage satisfied repeat customers."
    )

# 10. Opportunities / further investigation
missing_pieces = [role for role, col in detected.items() if col is None]
if missing_pieces:
    add_insight(
        f"The dataset does not contain columns for: {', '.join(missing_pieces)}.",
        "Column detection results (Section 2).",
        "Some standard retail KPIs and analyses could not be computed without this information.",
        "If available elsewhere, joining in additional data (e.g. cost/margin, customer "
        "demographics) would unlock deeper profitability and segmentation analysis."
    )

insights_df = pd.DataFrame(insights)
pd.set_option('display.max_colwidth', None)
insights_df


In [ ]:
# Print insights in a readable narrative format
for i, row in insights_df.iterrows():
    print(f"\n{'='*70}\nINSIGHT {i+1}")
    print(f"Finding: {row['Finding']}")
    print(f"Evidence: {row['Evidence']}")
    print(f"Business Interpretation: {row['Business Interpretation']}")
    print(f"Suggested Action: {row['Suggested Action']}")
print(f"\n{'='*70}")


---
## 8. Interactive Question-Answering System

Ask questions about your dataset in plain English using the text box below. This uses a
**rule-based** system (no external API required) that matches your question to the metrics
already calculated in this notebook. If a question can't be matched to an available metric,
it will tell you honestly rather than inventing an answer.

**Example questions you can try:**
- What is the total revenue?
- Which product category generated the highest revenue?
- What is the average order value?
- Which month had the highest sales?
- What is the monthly sales growth rate?
- Which sales channel generated the most revenue?
- What is the relationship between discounts and revenue?
- Which transactions are outliers?
- What are the main insights?
- Compare revenue across product categories.
- What percentage of revenue comes from each channel?


In [ ]:
import re

def answer_question(question):
    """Rule-based Q&A engine. Returns a text answer and optionally displays a table/chart."""
    q = question.lower().strip()

    # Total revenue
    if 'total revenue' in q or ('overall' in q and 'revenue' in q):
        if total_revenue is not None:
            return f"Total revenue is {total_revenue:,.2f}."
        return "Total revenue cannot be calculated — no revenue column was detected."

    # Highest revenue category
    if 'category' in q and ('highest' in q or 'top' in q or 'best' in q):
        if detected['category'] and rc:
            top = cat_table['total_revenue'].idxmax()
            return (f"'{top}' generated the highest revenue: {cat_table.loc[top, 'total_revenue']:,.2f} "
                    f"({cat_table.loc[top, 'revenue_contribution_pct']:.1f}% of total revenue).")
        return "Cannot answer — no product category column was detected."

    # Compare categories
    if 'compare' in q and 'categor' in q:
        if detected['category'] and rc:
            display(cat_table.sort_values('total_revenue', ascending=False))
            return "Category comparison table displayed above."
        return "Cannot answer — no product category column was detected."

    # Lowest / worst category
    if 'category' in q and ('lowest' in q or 'worst' in q or 'underperform' in q):
        if detected['category'] and rc:
            bottom = cat_table['total_revenue'].idxmin()
            return (f"'{bottom}' generated the lowest revenue: {cat_table.loc[bottom, 'total_revenue']:,.2f}.")
        return "Cannot answer — no product category column was detected."

    # AOV
    if 'average order value' in q or 'aov' in q:
        if aov is not None:
            return f"The Average Order Value (AOV) is {aov:,.2f}."
        return "Cannot answer — Total Revenue and/or Total Orders are unavailable."

    # Total orders
    if 'total orders' in q or 'how many orders' in q or 'number of orders' in q:
        if total_orders:
            return f"Total number of orders: {total_orders:,}."
        return "Cannot answer — no order/transaction ID column was detected."

    # Total units
    if 'units' in q and ('total' in q or 'how many' in q or 'sold' in q):
        if total_units is not None:
            return f"Total units sold: {total_units:,}."
        return "Cannot answer — no quantity column was detected."

    # Highest/best sales month
    if 'month' in q and ('highest' in q or 'best' in q or 'peak' in q or 'top' in q):
        if dtc:
            return f"The month with the highest revenue was {monthly_rev.idxmax()} ({monthly_rev.max():,.2f})."
        return "Cannot answer — no date column was detected."

    # Lowest sales month
    if 'month' in q and ('lowest' in q or 'worst' in q or 'weak' in q):
        if dtc:
            return f"The month with the lowest revenue was {monthly_rev.idxmin()} ({monthly_rev.min():,.2f})."
        return "Cannot answer — no date column was detected."

    # Growth rate
    if 'growth' in q:
        if dtc and growth_table is not None:
            display(growth_table)
            return (f"Average month-over-month growth rate: {growth_table['pct_growth'].mean():.2f}%. "
                    f"See the table displayed above for the month-by-month breakdown.")
        return "Cannot answer — sales growth rate requires a date column and revenue column."

    # Channel with most revenue
    if 'channel' in q and ('most' in q or 'highest' in q or 'best' in q or 'top' in q):
        if detected['channel'] and rc:
            top_ch = chan_table['total_revenue'].idxmax()
            return f"'{top_ch}' generated the most revenue: {chan_table.loc[top_ch, 'total_revenue']:,.2f}."
        return "Cannot answer — no sales channel column was detected."

    # Channel revenue percentage
    if 'channel' in q and ('percent' in q or '%' in q or 'contribution' in q or 'share' in q):
        if detected['channel'] and rc:
            display(chan_table[['total_revenue', 'revenue_contribution_pct']])
            return "Channel revenue contribution table displayed above."
        return "Cannot answer — no sales channel column was detected."

    # Discount vs revenue
    if 'discount' in q and 'revenue' in q:
        if dc and rc:
            return (f"The Pearson correlation between discount and revenue is {pearson_corr:.3f} "
                    f"and the Spearman correlation is {spearman_corr:.3f}. This suggests a "
                    f"{'weak' if abs(pearson_corr) < 0.3 else 'moderate-to-strong'} relationship. "
                    f"Remember: correlation does not imply causation.")
        return "Cannot answer — no discount and/or revenue column was detected."

    # Outliers
    if 'outlier' in q:
        if rc:
            return (f"{len(outliers):,} transactions were flagged as outliers using the IQR method "
                    f"(outside {lower_bound:,.2f} to {upper_bound:,.2f} on '{rc}').")
        return "Cannot answer — no revenue column was detected."

    # Insights
    if 'insight' in q:
        for i, row in insights_df.iterrows():
            print(f"{i+1}. {row['Finding']}")
        return "Full insights (with evidence and suggested actions) are listed in Section 7 above."

    # Customer-related
    if 'customer' in q:
        if detected['customer_id'] and rc:
            return (f"There are {unique_customers:,} unique customers, {repeat_customers:,} of whom "
                    f"({repeat_customers/unique_customers*100:.1f}%) are repeat customers. "
                    f"Average revenue per customer is {avg_spend_per_customer:,.2f}.")
        return "Cannot answer — no customer ID column was detected."

    return ("I'm not able to match that question to a metric this notebook calculated. "
            "Try rephrasing it, or check the section headings above for what's available "
            "(revenue, orders, AOV, units, growth, categories, channels, discounts, outliers, customers).")

print("✅ Q&A engine ready. Try running the cell below with your own question, "
      "or use the interactive text box in the next cell.")


In [ ]:
# --- Try it out directly ---
sample_question = "Which product category generated the highest revenue?"
print(f"Q: {sample_question}")
print(f"A: {answer_question(sample_question)}")


In [ ]:
# --- Interactive widget (works in Google Colab) ---
try:
    import ipywidgets as widgets
    from IPython.display import display as ip_display

    question_box = widgets.Text(
        value='',
        placeholder='Type your question here, e.g. "What is the total revenue?"',
        description='Question:',
        layout=widgets.Layout(width='600px')
    )
    ask_button = widgets.Button(description='Ask', button_style='primary')
    output_area = widgets.Output()

    def on_ask_clicked(b):
        with output_area:
            output_area.clear_output()
            q = question_box.value
            if not q.strip():
                print("Please type a question first.")
                return
            print(f"Q: {q}")
            print(f"A: {answer_question(q)}")

    ask_button.on_click(on_ask_clicked)
    ip_display(widgets.HBox([question_box, ask_button]), output_area)
except ImportError:
    print("ipywidgets not available in this environment — use the answer_question('your question') "
          "function directly instead, e.g.:\n  answer_question('What is the total revenue?')")


### 8.1 (Optional) LLM-Powered Free-Form Q&A

The rule-based system above works without any API key. If you'd like to ask more open-ended
questions using an LLM, you can optionally enable this by supplying your own Anthropic API key below.
**This section is entirely optional — the notebook works fully without it, and no key is ever
printed or stored in notebook output.**


In [ ]:
# Optional: LLM-powered Q&A using your own Anthropic API key.
# Leave ENABLE_LLM_QA = False to skip this section entirely.

ENABLE_LLM_QA = False  # set to True and provide your key via getpass below to use this

if ENABLE_LLM_QA:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key (input hidden, not stored/printed): ")

    try:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)

        def llm_answer(question, context_summary):
            response = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=500,
                messages=[{
                    "role": "user",
                    "content": (
                        f"You are analyzing a retail sales dataset. Here is a summary of the "
                        f"pre-calculated metrics:\n{context_summary}\n\n"
                        f"Answer this question using ONLY the information above. If the answer "
                        f"isn't in the data provided, say so clearly:\n{question}"
                    )
                }]
            )
            return response.content[0].text

        # Build a compact context summary from what we already calculated
        context_summary = kpi_summary.to_string()
        print("✅ LLM Q&A ready. Example usage:")
        print("   llm_answer('What stands out most in this data?', context_summary)")
    except ImportError:
        print("The 'anthropic' package is not installed. Run: !pip install anthropic")
    del api_key  # never keep the key in memory longer than needed
else:
    print("LLM-powered Q&A is disabled (ENABLE_LLM_QA = False). "
          "The rule-based Q&A system above already covers the standard analytical questions.")


---
## 9. Final Analytical Report

The cell below auto-generates a structured business report from everything calculated in this
notebook. Re-run it any time after re-running the analysis above (e.g. with a new dataset).


In [ ]:
report_lines = []
report_lines.append("# 📊 RETAIL SALES ANALYSIS — FINAL REPORT")
report_lines.append(f"\n*Dataset: {selected_file} | {len(df_raw):,} raw rows → {len(df):,} cleaned rows*\n")

# --- Executive Summary ---
report_lines.append("## Executive Summary")
exec_summary = []
if total_revenue is not None:
    exec_summary.append(f"the business generated {total_revenue:,.2f} in total revenue")
if total_orders:
    exec_summary.append(f"across {total_orders:,} orders")
if aov is not None:
    exec_summary.append(f"at an average order value of {aov:,.2f}")
if dtc:
    exec_summary.append(f"spanning {df[dtc].min().date()} to {df[dtc].max().date()}")
report_lines.append("Over the period covered by this dataset, " + ", ".join(exec_summary) + ".")

# --- Dataset Quality ---
report_lines.append("\n## Dataset Quality")
report_lines.append(f"- Original rows: {original_row_count:,} → Final cleaned rows: {final_row_count:,} "
                     f"({original_row_count - final_row_count:,} removed).")
for entry in cleaning_log:
    report_lines.append(f"- {entry}")

# --- KPI Summary ---
report_lines.append("\n## KPI Summary")
for _, row in kpi_summary.iterrows():
    report_lines.append(f"- **{row['KPI']}**: {row['Value']} — {row['Interpretation']}")

# --- Key Findings ---
report_lines.append("\n## Key Findings")
for _, row in insights_df.iterrows():
    report_lines.append(f"- {row['Finding']}")

# --- Business Opportunities ---
report_lines.append("\n## Business Opportunities")
for _, row in insights_df.iterrows():
    report_lines.append(f"- {row['Suggested Action']}")

# --- Limitations ---
report_lines.append("\n## Limitations")
missing_pieces = [role for role, col in detected.items() if col is None]
if missing_pieces:
    report_lines.append(f"- The following expected data elements were not found in this dataset: "
                         f"{', '.join(missing_pieces)}. Related KPIs/analyses were skipped.")
else:
    report_lines.append("- No expected data elements were missing from this dataset.")
if dtc:
    report_lines.append(f"- Date coverage: {df[dtc].min().date()} to {df[dtc].max().date()}. "
                         f"Any gaps within this range were not separately validated.")
report_lines.append("- All correlation-based findings (e.g. discount vs. revenue) describe association, "
                     "not causation.")
report_lines.append("- Outliers were flagged, not removed, so extreme values still influence aggregate KPIs.")

# --- Conclusion ---
report_lines.append("\n## Conclusion")
report_lines.append(
    "This dataset shows a retail business with measurable, quantifiable performance across "
    "products, channels, and customers. The KPIs and insights above are fully derived from the "
    "uploaded data and should be used as a starting point for deeper investigation — particularly "
    "around the top/bottom category gap, channel mix, and customer retention — rather than as "
    "final conclusions on their own."
)

final_report = "\n".join(report_lines)
from IPython.display import Markdown, display as ip_display
ip_display(Markdown(final_report))


In [ ]:
# Optionally save the report to a text/markdown file you can download
with open('final_analytical_report.md', 'w') as f:
    f.write(final_report)

print("✅ Report saved to 'final_analytical_report.md'.")

try:
    from google.colab import files as colab_files
    colab_files.download('final_analytical_report.md')
except ImportError:
    print("(Not running in Colab — file saved locally in the working directory instead of auto-downloading.)")


---
## 🎉 End of Notebook

You've now run a complete retail sales analysis: data cleaning, EDA, KPIs, business performance
analysis, evidence-based insights, and an interactive Q&A system — all generated automatically
from whatever dataset you uploaded.

**To analyze a different dataset:** re-run this notebook from the top and upload a new file in
Section 1. Because column detection is keyword-based rather than hardcoded, most standard retail
sales datasets should work without any code changes.
